# Pipeline B — mid fusion + painted LiDAR — results

**Presentation notebook — loads the trained artifacts and renders results; it does not train.** Produce them first via `training.ipynb` with `MODEL = "pipeline_b"`.

## 0. Setup

In [ ]:
# Repo-root bootstrap — lets this notebook run from notebooks/ or the repo root.
import os, sys
from pathlib import Path
_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "globals.py").exists())
os.chdir(_ROOT)
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
print("repo root:", _ROOT)

In [ ]:
import os
from pathlib import Path
import data  # noqa: F401
_REPO = Path(data.__file__).resolve().parent
os.environ.setdefault("PY123D_DATA_ROOT", str(_REPO / "data"))
os.environ.setdefault("KITTI360_DATA_ROOT", str(_REPO / "KITTI-360"))

import json
import matplotlib.pyplot as plt
import torch

import globals as G
import utils
from data import Py123dDataset, stereo_cache_root
from evaluation import CenterPointDecoder, load_report, print_ap_report
from network import StereoBEVConfig, build_detector
import igev_matcher  # noqa: F401

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL = "pipeline_b"
TAG = "_yolo26_igev"
CKPT = f"checkpoints/{MODEL}{TAG}.pt"
RESULTS = f"results/{MODEL}{TAG}.json"
HISTORY = f"results/{MODEL}{TAG}_history.json"
FIGDIR = Path("docs/img"); FIGDIR.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "| model:", MODEL, "| classes:", G.CLASSES)

## 1. Training loss curves

In [ ]:
if not Path(HISTORY).exists():
    print(f"no {HISTORY} — (re)run training.ipynb §6 with the current notebook to "
          "save the loss curves (§6 calls save_history).")
else:
    history = json.loads(Path(HISTORY).read_text())
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(history["steps"], alpha=0.6); ax[0].set_yscale("log")
    ax[0].set_title(f"{MODEL} — per-step loss"); ax[0].set_xlabel("step"); ax[0].grid(alpha=0.25)
    ax[1].plot(history["train"], "o-", label="train"); ax[1].plot(history["val"], "s-", label="val")
    ax[1].set_title(f"{MODEL} — per-epoch mean loss"); ax[1].set_xlabel("epoch")
    ax[1].legend(); ax[1].grid(alpha=0.25)
    fig.tight_layout(); fig.savefig(FIGDIR / f"{MODEL}{TAG}_loss.png", dpi=90, bbox_inches="tight")
    plt.show()

## 2. Detection AP (per class · 0.5/1/2/4 m)

In [ ]:
report = load_report(RESULTS)
print_ap_report(report)
print(f"\nHEADLINE  mAP {report['mAP']:.3f}  |  macro F1 {report['f1']:.3f} "
      f"@{report['op_threshold_m']:g} m  |  mean centre err {report['mean_error_m']:.3f} m")

## 3. Diagnostics — PR, F1-vs-confidence, confusion

In [ ]:
utils.visualize_evaluation(report, save_path=str(FIGDIR / f"{MODEL}{TAG}_diagnostics.png"))
utils.visualize_evaluation(report)

## 4. Qualitative — detections vs GT (BEV)

In [ ]:
val_ds = Py123dDataset(split_names=["kitti360_val"]); val_frames = list(val_ds)
CACHE = stereo_cache_root(val_ds.data_root, matcher="igev")
stereo_cfg = StereoBEVConfig(img_backbone="yolo26", yolo_freeze=True)
model, input_fn = build_detector(MODEL, stereo_cache_root=CACHE, stereo_cfg=stereo_cfg)
model.load_state_dict(torch.load(CKPT, weights_only=True)["model"]); model.to(DEVICE).eval()
decoder = CenterPointDecoder(score_threshold=0.2)
for i, frame in enumerate(val_frames[20:400:90]):
    sample = frame.to_stereo_sample(load_images=False, point_mask=False)
    with torch.no_grad():
        out = model(input_fn(sample) if input_fn else sample, device=DEVICE)
    det = decoder(out["heatmap"].cpu(), out["offset"].cpu())[0]
    utils.visualize_detections(sample, det)                                   # inline
    utils.visualize_detections(sample, det,                                   # + save for slides
        save_path=str(FIGDIR / f"{MODEL}{TAG}_det_{i}.png"))
print("saved qualitative BEV frames ->", FIGDIR)

## 5. Branch contribution (drop-branch ablation)

In [ ]:
from evaluation import evaluate_model
for drop in (None, "camera", "lidar"):
    model.drop_branch = drop
    rep = evaluate_model(model, val_frames, input_fn=input_fn, device=DEVICE)
    print(f"drop_branch={str(drop):<7} -> mAP {rep['mAP']:.3f}")
model.drop_branch = None

---
*Figures in `docs/img/`. Cross-model comparison: `confronto.ipynb`.*